# YT Agent — Colab GPU worker

Manual companion to the Kaggle GPU worker. Boots a FastAPI backend on a free Colab GPU runtime and connects **outbound** to your Coolify dashboard — no inbound tunnel, no static IP required. Once running it polls `/api/jobs/claim` and heartbeats every 30 sec into the Pocketbase `backends` collection, so the Monitor page detects it within milliseconds.

## Before you start
1. **Runtime → Change runtime type → T4 GPU** (free).
2. Open **🔑 Secrets** (left sidebar key icon) and add the **required** secrets:
   - `COOLIFY_BASE_URL` — e.g. `https://yt-agent.thyker.online`
   - `PB_URL` — e.g. `https://yt-agent.thyker.online/pb`
   - `POCKETBASE_ADMIN_EMAIL` + `POCKETBASE_ADMIN_PASSWORD` — same values as your Coolify env
   - `RENDER_TRIGGER_KEY` — shared secret for `/api/workers/register`
   - `STORAGE_PROVIDERS_ENC_KEY` — 32-byte hex; must match your dashboard

   NIM / Groq / Pexels / HuggingFace / YouTube OAuth all live in Pocketbase's `api_keys` collection — manage them via the dashboard's **Connections** page. This notebook never touches them.
3. **Toggle notebook access ON** for each secret (the X icons → green dots).
4. Run cells top to bottom. **Cell 4 is the GPU diagnostic** — make sure it prints `READY: h264_nvenc` before proceeding; otherwise the render falls back to CPU and will be slow.
5. The last cell is a blocking server loop — leave it running.

In [ ]:
# 1) System deps
import subprocess, sys, os
subprocess.check_call(["apt-get", "-qq", "update"])
# fonts-noto{,-cjk,-color-emoji} + fonts-dejavu-core — needed for
# burned-in captions to render non-Latin scripts (Devanagari, Bengali,
# CJK, Arabic, Thai). DejaVu Sans is the primary font (bundled
# default); libass falls back to Noto CJK for CJK/emoji.
subprocess.check_call(["apt-get", "-qq", "install", "-y",
                       "ffmpeg", "fonts-dejavu-core",
                       "fonts-noto", "fonts-noto-cjk",
                       "fonts-noto-color-emoji"])
subprocess.run(["fc-cache", "-f"], check=False)
print("ffmpeg:", subprocess.check_output(["ffmpeg", "-version"]).decode().splitlines()[0])

In [ ]:
# 2) Clone the repo (or pull latest if already present)
REPO_URL = "https://github.com/Ahsan3301/yt_agent.git"
BRANCH   = "main"
import os, subprocess
if not os.path.exists('/content/yt_agent'):
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, '/content/yt_agent'])
else:
    subprocess.check_call(['git', '-C', '/content/yt_agent', 'pull', '--ff-only'])
os.chdir('/content/yt_agent')
print('repo at:', os.getcwd())
print('commit:', subprocess.check_output(['git', 'log', '-1', '--oneline']).decode().strip())

In [ ]:
# 3) Python deps — base + GPU extras + browser agent + Kokoro TTS
#
# - requirements.txt: core (fastapi, boto3, paramiko, pocketbase, edge-tts>=7, ...)
# - decord + av: GPU decode/encode for editor_gpu (torch is preinstalled
#   in the Colab runtime; do NOT reinstall it, that clobbers CUDA).
# - Kokoro TTS: needs phonemizer-fork (has EspeakWrapper.set_data_path
#   which vanilla `phonemizer` lacks) + espeakng-loader (bundles the
#   data path misaki asks for) + the espeak-ng apt binary. Without any
#   one of these Kokoro raises at import time and we fall back to edge-tts.
# - edge-tts pinned to >=7.0 because Microsoft changed the auth endpoint
#   in early 2025; older wheels return NoAudioReceived and give us
#   silent failures for every voiceover.
# - requirements-browser.txt + playwright chromium: NIM-driven research
#   agent (fact-checks topics + grabs hero images).
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'decord==0.6.0', 'av==12.3.0'])
# hf_transfer — Rust-based parallel HTTPS downloader for HuggingFace.
# 3-5x faster on any HF fetch. Cuts SDXL first-load (~7 GB) from
# ~4 min to ~1 min, and Kokoro's ~330 MB model too. Env var must
# be set BEFORE the diffusers/huggingface_hub imports so the fast
# backend is selected.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'hf_transfer'])
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

# Install diffusers WITHOUT letting it upgrade torch — Colab ships
# a torch wheel with sm_60/sm_75 kernels for T4; the latest torch
# on PyPI has dropped some older archs and would silently clobber
# the working install, causing cudaErrorNoKernelImageForDevice at
# pipe.to('cuda'). --no-deps + explicit deps preserves the runtime.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'diffusers>=0.30'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.40', 'accelerate>=0.30', 'safetensors>=0.4', 'huggingface_hub>=0.23', 'peft>=0.10'])
subprocess.check_call(['apt-get', 'install', '-y', '-qq', 'espeak-ng'])
# Order matters: install phonemizer-fork BEFORE kokoro so pip does not
# also pull vanilla phonemizer as a transitive dep on top of it.
# Kaggle's / Colab's base image ships vanilla `phonemizer` — its
# EspeakWrapper doesn't have set_data_path(), which is what misaki
# (Kokoro's phoneme dep) calls at import. If we leave it installed,
# pip's install of phonemizer-fork lands ALONGSIDE and the older one
# wins on `import phonemizer`. Uninstall it FIRST, then install the
# fork with --force-reinstall so the correct wheel definitely lands.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'phonemizer'], check=False)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', 'phonemizer-fork>=3.3.2'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'espeakng-loader>=0.2'])
# Kokoro TTS + misaki (its phoneme dep) + soundfile (Kokoro's audio
# writer). Kaggle notebook has the same block; Colab was missing it,
# so import kokoro failed and every render silently fell back to
# edge-tts. Kokoro-82M produces materially higher quality on English.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'soundfile>=0.12',
    'kokoro>=0.9.4',
    'misaki>=0.9.4',
])
# Force-reinstall edge-tts in case requirements.txt already picked an
# older cached wheel (Colab runtimes sometimes carry 6.x in the base image).
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'edge-tts>=7.0.0'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-browser.txt'])
# --with-deps: apt-installs the shared libs Chromium needs (libnss3,
# libatk*, libgbm, etc). Without them the download succeeds but
# chromium.launch() raises 'error while loading shared libraries'
# and research_agent silently falls back to text-only research.
subprocess.check_call([sys.executable, '-m', 'playwright', 'install', '--with-deps', 'chromium'])
import torch
print('torch:', torch.__version__, 'cuda available:', torch.cuda.is_available())
try:
    from kokoro import KPipeline
    print('kokoro: ready')
except Exception as e:
    print('kokoro: NOT ready ->', e)
try:
    import edge_tts
    print('edge-tts:', edge_tts.__version__ if hasattr(edge_tts, '__version__') else 'installed')
except Exception as e:
    print('edge-tts: NOT ready ->', e)
print('playwright + chromium ready — NIM research agent will activate on fact_research channels')
try:
    from diffusers import AutoPipelineForText2Image  # noqa: F401
    print('diffusers: ready (local_sdxl provider available)')
except Exception as e:
    print('diffusers: NOT ready ->', e)


In [ ]:
# 4) GPU / NVENC diagnostic
#
# Three checks decide whether the pipeline will render on GPU (5-10x
# faster) or fall back to CPU:
#   A. nvidia-smi sees a GPU
#   B. the installed ffmpeg was built with h264_nvenc support
#   C. a live NVENC smoke encode actually succeeds at runtime
#
# Note on the smoke size: NVENC has a minimum frame size (~145x49 for
# H.264). The previous 64x64 test failed even on working T4s with
# "Frame Dimension less than the minimum supported value" — that's a
# test bug, not a GPU problem. 320x240 is safely above the minimum and
# tells the truth about whether your render pipeline will work.
#
# If you see 'READY: h264_nvenc' at the bottom, the backend's encoder
# picker (modules/editor._detect_nvenc) will pick GPU on startup and
# the Monitor card will show 'renders: GPU ✓'.
import subprocess, shutil, os

print('A) nvidia-smi -L')
if shutil.which('nvidia-smi') is None:
    print('   FAIL: nvidia-smi not on PATH — runtime is probably CPU. Runtime → Change runtime type → T4 GPU.')
    gpu_ok = False
else:
    r = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, timeout=8)
    print('  ', (r.stdout or '').strip() or r.stderr.strip())
    gpu_ok = (r.returncode == 0 and 'GPU' in (r.stdout or ''))

print()
print('B) ffmpeg -encoders | grep h264_nvenc')
r = subprocess.run(['ffmpeg', '-hide_banner', '-encoders'], capture_output=True, text=True, timeout=8)
ffmpeg_ok = 'h264_nvenc' in (r.stdout or '')
print('   PASS' if ffmpeg_ok else '   FAIL: the installed ffmpeg has no h264_nvenc encoder — install a CUDA-enabled build')

print()
print('C) live NVENC smoke encode (320x240 black, 0.1s)')
smoke_ok = False
if gpu_ok and ffmpeg_ok:
    r = subprocess.run([
        'ffmpeg', '-hide_banner', '-loglevel', 'error',
        '-f', 'lavfi', '-i', 'color=c=black:s=320x240:d=0.1',
        '-c:v', 'h264_nvenc', '-f', 'null', '-',
    ], capture_output=True, text=True, timeout=15)
    smoke_ok = (r.returncode == 0)
    if smoke_ok:
        print('   PASS')
    else:
        print('   FAIL:', (r.stderr or '').strip()[-300:])
else:
    print('   SKIPPED (A or B failed)')

print()
if gpu_ok and ffmpeg_ok and smoke_ok:
    print('READY: h264_nvenc — pipeline will render on GPU')
    os.environ.pop('FFMPEG_FORCE_CPU', None)
else:
    print('READY: libx264 (CPU only) — to fix, change runtime to T4 GPU and run all cells again')
    # We don't set FFMPEG_FORCE_CPU here — the backend's auto-detect
    # falls back to libx264 on its own. Setting it would also stop
    # any later in-place fixes from being picked up after restart.


In [ ]:
# 5) Bootstrap secrets.
#
#    REQUIRED (Pocketbase + Coolify mode — the current default):
#      COOLIFY_BASE_URL             (dashboard URL, e.g. https://yt-agent.thyker.online)
#      PB_URL                       (Pocketbase URL, usually <dashboard>/pb)
#      POCKETBASE_ADMIN_EMAIL       (superuser email)
#      POCKETBASE_ADMIN_PASSWORD    (superuser password)
#      RENDER_TRIGGER_KEY           (shared secret for /api/workers/register etc.)
#      STORAGE_PROVIDERS_ENC_KEY    (32-byte hex, same value as your dashboard)
#
#    OPTIONAL (legacy Firestore path — only if your dashboard is still
#    on Vercel + Firebase):
#      GOOGLE_APPLICATION_CREDENTIALS_JSON      (raw JSON)
#      GOOGLE_APPLICATION_CREDENTIALS_JSON_B64  (base64 alt — mirrors Kaggle's Secrets workaround)
#      R2_* / SFTP_* / PUBLIC_BASE_URL          (used if no PB storage provider configured)
#
#    NIM / Groq / Pexels / HuggingFace etc. all live in Pocketbase's
#    api_keys collection — manage via the dashboard's Connections page.
from google.colab import userdata
import os

REQUIRED = [
    'COOLIFY_BASE_URL',
    'PB_URL',
    'POCKETBASE_ADMIN_EMAIL',
    'POCKETBASE_ADMIN_PASSWORD',
    'RENDER_TRIGGER_KEY',
    'STORAGE_PROVIDERS_ENC_KEY',
]
OPTIONAL = [
    'GOOGLE_APPLICATION_CREDENTIALS_JSON',
    'GOOGLE_APPLICATION_CREDENTIALS_JSON_B64',
    'R2_ACCOUNT_ID', 'R2_ACCESS_KEY_ID', 'R2_SECRET_ACCESS_KEY',
    'R2_BUCKET', 'R2_PUBLIC_URL', 'R2_MAX_GB',
    'SFTP_HOST', 'SFTP_PORT', 'SFTP_USER', 'SFTP_PASS', 'SFTP_BASE_DIR',
    'PUBLIC_BASE_URL',
    'INSTANCE_LABEL', 'INSTANCE_TIER',
]

missing = []
for k in REQUIRED + OPTIONAL:
    try:
        v = userdata.get(k)
        if v:
            os.environ[k] = str(v)
    except Exception:
        if k in REQUIRED:
            missing.append(k)
        continue
    if k in REQUIRED and not os.environ.get(k):
        missing.append(k)

# Defaults for the outbound-poll model.
os.environ.setdefault('DB_BACKEND', 'pocketbase')
os.environ.setdefault('STORAGE_BACKEND', 'registry')
os.environ.setdefault('WORKER_MODE', 'outbound_poll')
os.environ.setdefault('INSTANCE_TIER', 'gpu')
os.environ.setdefault('INSTANCE_LABEL', 'colab-gpu')
# Idle auto-shutdown — releases the GPU sooner so the next render gets
# a fresh runtime. Same value the Kaggle worker uses (25 min).
os.environ.setdefault('KAGGLE_AUTO_SHUTDOWN_AFTER_IDLE_SECONDS', '1500')

if missing:
    print('MISSING REQUIRED BOOTSTRAP SECRETS:', ', '.join(missing))
    print('Add them in the Secrets panel and toggle notebook access ON, then re-run this cell.')
else:
    print('Bootstrap secrets loaded.')
    print('  Pocketbase     :', os.environ.get('PB_URL'))
    print('  Dashboard API  :', os.environ.get('COOLIFY_BASE_URL'))
    print('  Worker mode    :', os.environ.get('WORKER_MODE'))
    print('  DB backend     :', os.environ.get('DB_BACKEND'))
    print('  Storage backend:', os.environ.get('STORAGE_BACKEND'))
    print('  Tier / label   :', os.environ.get('INSTANCE_TIER'), '/', os.environ.get('INSTANCE_LABEL'))
    print('  Idle shutdown  :', os.environ.get('KAGGLE_AUTO_SHUTDOWN_AFTER_IDLE_SECONDS'), 's')
    print('  Storage providers + API keys will be pulled from Pocketbase.')

In [ ]:
# 6) (Skipped in outbound-poll mode.) In WORKER_MODE=outbound_poll the
#    worker calls the dashboard from inside Colab — no inbound tunnel
#    is needed. The cell below is kept for legacy WORKER_MODE=tunnel
#    deployments (Vercel + Firestore) and is a no-op otherwise.
import os, subprocess, time, re

if os.environ.get('WORKER_MODE', 'tunnel').lower() != 'tunnel':
    print('outbound-poll mode — no cloudflared tunnel needed.')
else:
    if not os.path.exists('/usr/local/bin/cloudflared'):
        subprocess.check_call([
            'wget', '-q',
            'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
            '-O', '/usr/local/bin/cloudflared',
        ])
        subprocess.check_call(['chmod', '+x', '/usr/local/bin/cloudflared'])
    tunnel_log = '/tmp/cloudflared.log'
    subprocess.Popen(
        ['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:8000',
         '--logfile', tunnel_log, '--loglevel', 'info'],
        stdout=open(tunnel_log + '.stdout', 'w'),
        stderr=subprocess.STDOUT,
    )
    url = None
    for _ in range(40):
        time.sleep(0.5)
        if not os.path.exists(tunnel_log):
            continue
        with open(tunnel_log) as f:
            m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', f.read())
            if m:
                url = m.group(0)
                break
    if not url:
        raise RuntimeError('cloudflared did not produce a URL — check /tmp/cloudflared.log')
    os.environ['PUBLIC_BACKEND_URL'] = url
    print('Public backend URL:', url)


In [ ]:
# 6.5) Pre-warm SDXL model download + Kokoro touch (boot time).
#
# Without this the FIRST job pays the full SDXL download+materialize
# and Kokoro cold-start. Pre-downloading into the HF cache lets
# subsequent .from_pretrained() calls skip the network step.
import os, time, subprocess
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')

try:
    t0 = time.time()
    print('preboot: snapshotting SDXL weights into HF cache...')
    from huggingface_hub import snapshot_download
    _sdxl_model = os.environ.get('LOCAL_SDXL_MODEL', 'stabilityai/sdxl-turbo')
    snapshot_download(_sdxl_model, allow_patterns=['*.json', '*.txt', '*.safetensors'])
    print(f'preboot: SDXL weights cached in {time.time()-t0:.1f}s')
except Exception as e:
    print(f'preboot: SDXL cache warm skipped ({e!r})')

try:
    t1 = time.time()
    from kokoro import KPipeline
    _kp = KPipeline(lang_code='a')
    for _ in _kp('warmup', voice='am_michael', speed=1.0):
        break
    del _kp
    print(f'preboot: Kokoro warm in {time.time()-t1:.1f}s')
except Exception as e:
    print(f'preboot: Kokoro warm skipped ({e!r})')

try:
    out = subprocess.check_output(['nvidia-smi', '-L'], timeout=5).decode().strip()
    print('preboot: nvidia-smi -L:\n' + out)
except Exception as e:
    print(f'preboot: nvidia-smi -L failed ({e!r})')

In [ ]:
# 6.6) Self-check — verify multi-GPU + multilingual wiring
#
# Same as Kaggle cell 4.6. Runs backend/self_check.py; any FAIL is
# worth fixing before rendering. Full report in <1 second.
from backend import self_check
print(self_check.run(text=True))

In [ ]:
# 7) Boot the backend. This cell BLOCKS — keep it running for the session.
#    In outbound-poll mode the worker POSTs to <COOLIFY_BASE_URL>/api/workers/register
#    on boot and heartbeats every 30s so the Monitor detects it. It then
#    polls /api/jobs/claim for queued renders. Watch the first
#    'video encoder: ...' log line — it tells you whether NVENC was picked.
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'uvicorn', 'backend.server:app',
    '--host', '0.0.0.0', '--port', '8000',
    '--log-level', 'info',
])